In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from keras.layers import TextVectorization, Input, Embedding, SpatialDropout1D, LSTM, Dense, Bidirectional, Dropout
from keras.models import Model

In [3]:
MAX_VOCAB_SIZE = 30000
EMBEDDING_DIM = 100
MAX_SEQUENCE_LENGTH = 300
PADDING_TYPE = 'post'
TRUNC_TYPE = 'post'
OOV_TOK = '<OOV>'
GLOVE_PATH = '/content/drive/MyDrive/Colab Notebooks/deep learning projects/Fake News Detection/glove.6B.100d.txt'
EPOCHS = 10
BATCH_SIZE = 32

In [4]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/deep learning projects/Fake News Detection/WELFake_Dataset.csv')
df = df.drop('Unnamed: 0', axis=1)
df.head()

,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,NaN,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [5]:
df.dropna(subset=['label'], inplace=True)
df['title'] = df['title'].fillna("")
df['text'] = df['text'].fillna("")

df['combined_text'] = df['title'] + ' ' + df['text']

X = df['combined_text'].values
y = df['label'].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
vectorizer = TextVectorization(
    max_tokens=MAX_VOCAB_SIZE,
    output_sequence_length=MAX_SEQUENCE_LENGTH,
    output_mode='int'
)
vectorizer.adapt(X_train)
vocab = vectorizer.get_vocabulary()
word_index = {word: index for index, word in enumerate(vocab)}

In [7]:
embedding_index = {}
if os.path.exists(GLOVE_PATH):
  with open(GLOVE_PATH, 'r', encoding='utf-8') as file:
      for line in file:
          word, coefs = line.split(maxsplit=1)
          coefs = np.fromstring(coefs, 'f', sep=' ')
          embedding_index[word] = coefs
else:
  raise FileNotFoundError(f"GloVe file not found at path: {GLOVE_PATH}")

num_tokens = len(vocab)
embedding_matrix = np.zeros((num_tokens, EMBEDDING_DIM))

for word, i in word_index.items():
  embedding_vector = embedding_index.get(word)
  if embedding_vector is not None:
    embedding_matrix[i] = embedding_vector

In [8]:
inputs = Input(shape=(1,), dtype=tf.string)
x = vectorizer(inputs)

embedding = Embedding(num_tokens, EMBEDDING_DIM, embeddings_initializer=embedding_matrix, trainable=False)
embedding.build((None,))
embedding.set_weights([embedding_matrix])

x = embedding(x)
x = SpatialDropout1D(0.3)(x)
x = Bidirectional(LSTM(64, return_sequences=False))(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.5)(x)

outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=EPOCHS, batch_size=BATCH_SIZE)
model.summary()

Epoch 1/10
1804/1804 ━━━━━━━━━━━━━━━━━━━━ 60s 30ms/step - accuracy: 0.9046 - loss: 0.2318 - val_accuracy: 0.9485 - val_loss: 0.1267
Epoch 2/10
1804/1804 ━━━━━━━━━━━━━━━━━━━━ 53s 29ms/step - accuracy: 0.9541 - loss: 0.1175 - val_accuracy: 0.9621 - val_loss: 0.0968
Epoch 3/10
1804/1804 ━━━━━━━━━━━━━━━━━━━━ 52s 29ms/step - accuracy: 0.9653 - loss: 0.0904 - val_accuracy: 0.9741 - val_loss: 0.0708
Epoch 4/10
1804/1804 ━━━━━━━━━━━━━━━━━━━━ 52s 29ms/step - accuracy: 0.9704 - loss: 0.0748 - val_accuracy: 0.9764 - val_loss: 0.0623
Epoch 5/10
1804/1804 ━━━━━━━━━━━━━━━━━━━━ 52s 29ms/step - accuracy: 0.9754 - loss: 0.0647 - val_accuracy: 0.9778 - val_loss: 0.0601
Epoch 6/10
1804/1804 ━━━━━━━━━━━━━━━━━━━━ 52s 29ms/step - accuracy: 0.9787 - loss: 0.0569 - val_accuracy: 0.9809 - val_loss: 0.0555
Epoch 7/10
1804/1804 ━━━━━━━━━━━━━━━━━━━━ 51s 29ms/step - accuracy: 0.9800 - loss: 0.0523 - val_accuracy: 0.9821 - val_loss: 0.0504
Epoch 8/10
1804/1804 ━━━━━━━━━━━━━━━━━━━━ 51s 28ms/step - accuracy: 0.9828 -

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization              │ (None, 300)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 300, 100)       │     3,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 300, 100)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        84,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,278,405 (12.51 MB)

 Trainable params: 92,801 (362.50 KB)

 Non-trainable params: 3,000,000 (11.44 MB)

 Optimizer params: 185,604 (725.02 KB)

In [9]:
model.save('/content/drive/MyDrive/Colab Notebooks/deep learning projects/Fake News Detection/fake_news_detection_model.keras')
print('Model saved successfully!')

Model saved successfully!
